# Graphs — LeetCode Questions (10 Qs)

Graph representations:
- Adjacency List: dict {node: [neighbors]}
- Grid as implicit graph: (r,c) → neighbors (up/down/left/right)

Core Algorithms:
- BFS: queue, level-by-level, shortest path in unweighted graph
- DFS: recursion or stack, cycle detection, connected components
- Topological Sort: Kahn's (BFS) or DFS for DAGs
- Union-Find: connected components, cycle detection

1.  200 – Number of Islands (DFS/BFS on grid)
2.  695 – Max Area of Island
3.  994 – Rotting Oranges (BFS)
4.  133 – Clone Graph
5.  207 – Course Schedule (Cycle Detection)
6.  210 – Course Schedule II (Topological Sort)
7.  417 – Pacific Atlantic Water Flow
8.  684 – Redundant Connection (Union-Find)
9.  743 – Network Delay Time (Dijkstra)
10. 130 – Surrounded Regions

In [ ]:
from typing import List, Optional
from collections import deque, defaultdict
import heapq

# 1. LeetCode 200 – Number of Islands

Count connected groups of '1's (land) in a grid. Up/down/left/right connections.

Input: grid=[[1,1,0],[0,1,0],[0,0,1]]  Output: 2

## Dry Run (DFS sink approach: mark visited as '0')

Grid:
```
1 1 0
0 1 0
0 0 1
```

| step | action | grid after | count |
|------|--------|------------|-------|
| (0,0)='1' | DFS: sink (0,0),(0,1),(1,1) | 0 0 0 / 0 0 0 / 0 0 1 | count=1 |
| (0,1)='0' | skip | | |
| (2,2)='1' | DFS: sink (2,2) | all zeros | count=2 |
| Result: **2** | | | |

In [ ]:
class Solution:
    def numIslands(self, grid: List[List[str]]) -> int:
        if not grid:
            return 0
        rows, cols = len(grid), len(grid[0])
        count = 0

        def dfs(r, c):
            if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] == '0':
                return
            grid[r][c] = '0'          # sink (mark visited)
            dfs(r+1, c); dfs(r-1, c)
            dfs(r, c+1); dfs(r, c-1)

        for r in range(rows):
            for c in range(cols):
                if grid[r][c] == '1':
                    count += 1
                    dfs(r, c)
        return count

grid = [['1','1','0'],['0','1','0'],['0','0','1']]
print(Solution().numIslands(grid))  # 2

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(m*n) – visit every cell at most once | O(m*n) – recursion stack worst case |
| **Final: O(m*n)** | **Final: O(m*n)** |

# 2. LeetCode 695 – Max Area of Island

Find the largest island (connected group of 1s) by cell count.

Input: [[0,0,1,0],[0,1,1,0],[0,1,0,0],[0,0,0,1]]  Output: 4

## Dry Run

DFS from each unvisited '1', count cells in that island.

| start cell | DFS explores | island size |
|------------|-------------|-------------|
| (0,2) | (0,2),(1,2),(1,1),(2,1) | 4 |
| (3,3) | (3,3) | 1 |
| max_area = **4** | | |

In [ ]:
class Solution:
    def maxAreaOfIsland(self, grid: List[List[int]]) -> int:
        rows, cols = len(grid), len(grid[0])
        max_area = 0

        def dfs(r, c):
            if r < 0 or r >= rows or c < 0 or c >= cols or grid[r][c] == 0:
                return 0
            grid[r][c] = 0            # mark visited
            return 1 + dfs(r+1,c) + dfs(r-1,c) + dfs(r,c+1) + dfs(r,c-1)

        for r in range(rows):
            for c in range(cols):
                if grid[r][c] == 1:
                    max_area = max(max_area, dfs(r, c))
        return max_area

grid = [[0,0,1,0],[0,1,1,0],[0,1,0,0],[0,0,0,1]]
print(Solution().maxAreaOfIsland(grid))  # 4

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(m*n) – every cell visited at most once | O(m*n) – recursion stack |
| **Final: O(m*n)** | **Final: O(m*n)** |

# 3. LeetCode 994 – Rotting Oranges

Every minute, fresh orange adjacent to rotten becomes rotten.
Return minutes to rot all, or -1 if impossible.

Input: [[2,1,1],[1,1,0],[0,1,1]]  Output: 4

Use Multi-Source BFS from all rotten oranges simultaneously.

## Dry Run (BFS from all rotten at once)

Grid: [[2,1,1],[1,1,0],[0,1,1]]
Initial rotten: [(0,0)], fresh=6

| minute | cells turned rotten | fresh left |
|--------|---------------------|------------|
| 0 | (0,0) already rotten | 6 |
| 1 | (0,1),(1,0) | 4 |
| 2 | (0,2),(1,1) | 2 |
| 3 | (2,1) | 1 |
| 4 | (2,2) | 0 → done |
| Result: **4** | | |

In [ ]:
class Solution:
    def orangesRotting(self, grid: List[List[int]]) -> int:
        rows, cols = len(grid), len(grid[0])
        queue = deque()
        fresh = 0

        for r in range(rows):
            for c in range(cols):
                if grid[r][c] == 2:
                    queue.append((r, c, 0))   # (row, col, minute)
                elif grid[r][c] == 1:
                    fresh += 1

        if fresh == 0:
            return 0

        minutes = 0
        dirs = [(1,0),(-1,0),(0,1),(0,-1)]
        while queue:
            r, c, t = queue.popleft()
            for dr, dc in dirs:
                nr, nc = r+dr, c+dc
                if 0<=nr<rows and 0<=nc<cols and grid[nr][nc]==1:
                    grid[nr][nc] = 2
                    fresh -= 1
                    minutes = t + 1
                    queue.append((nr, nc, t+1))

        return minutes if fresh == 0 else -1

print(Solution().orangesRotting([[2,1,1],[1,1,0],[0,1,1]]))  # 4
print(Solution().orangesRotting([[2,1,1],[0,1,1],[1,0,1]]))  # -1

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(m*n) – BFS visits every cell | O(m*n) – queue |
| **Final: O(m*n)** | **Final: O(m*n)** |

# 4. LeetCode 133 – Clone Graph

Deep copy of undirected graph. Each node has val and neighbors list.
Use DFS + hashmap {original: clone} to avoid revisiting.

Input: adjList=[[2,4],[1,3],[2,4],[1,3]]  Output: cloned graph

## Dry Run (DFS with visited map)

| DFS call | node | action | visited |
|----------|------|--------|---------|
| clone(1) | 1 | create Node(1), add to map | {1: Node(1)} |
| clone(2) | 2 | create Node(2) | {1,2} |
| clone(3) | 3 | create Node(3) | {1,2,3} |
| clone(4) | 4 | create Node(4) | {1,2,3,4} |
| all neighbors assigned via recursion | | | |

In [ ]:
class Node:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []

class Solution:
    def cloneGraph(self, node: Optional['Node']) -> Optional['Node']:
        if not node:
            return None
        visited = {}

        def dfs(n):
            if n in visited:
                return visited[n]
            clone = Node(n.val)
            visited[n] = clone             # store BEFORE recursing (avoid cycles)
            for neighbor in n.neighbors:
                clone.neighbors.append(dfs(neighbor))
            return clone

        return dfs(node)

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(V + E) – visit every node and edge | O(V) – visited hashmap + recursion stack |
| **Final: O(V + E)** | **Final: O(V)** |

# 5. LeetCode 207 – Course Schedule (Cycle Detection)

Can you finish all courses? True if no cycle in directed graph.
numCourses=2, prerequisites=[[1,0]] means: take 0 before 1.

Input: numCourses=2, prerequisites=[[1,0],[0,1]]  Output: False (cycle)

## Dry Run (DFS cycle detection with 3 states)

State: 0=unvisited, 1=visiting(in stack), 2=done

Graph: 0→1, 1→0 (cycle)

| DFS call | node | state before | action | cycle? |
|----------|------|-------------|--------|--------|
| dfs(0) | 0 | 0 | mark 1 (visiting) | |
| dfs(1) | 1 | 0 | mark 1 (visiting) | |
| dfs(0) | 0 | 1 (visiting!) | **CYCLE FOUND** | True |
| Result: cannot finish → **False** | | | | |

In [ ]:
class Solution:
    def canFinish(self, numCourses: int, prerequisites: List[List[int]]) -> bool:
        graph = defaultdict(list)
        for course, pre in prerequisites:
            graph[pre].append(course)

        # 0=unvisited, 1=visiting, 2=done
        state = [0] * numCourses

        def dfs(node):
            if state[node] == 1:
                return False    # cycle: already visiting this node
            if state[node] == 2:
                return True     # already fully processed, safe

            state[node] = 1                 # mark as visiting
            for neighbor in graph[node]:
                if not dfs(neighbor):
                    return False
            state[node] = 2                 # mark as done
            return True

        return all(dfs(i) for i in range(numCourses))

print(Solution().canFinish(2, [[1,0]]))       # True
print(Solution().canFinish(2, [[1,0],[0,1]])) # False (cycle)

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(V + E) – each node/edge visited once | O(V + E) – graph + state array + stack |
| **Final: O(V + E)** | **Final: O(V + E)** |

# 6. LeetCode 210 – Course Schedule II (Topological Sort)

Return an order to take all courses, or [] if cycle exists.
Topological sort = DFS postorder (add to result AFTER processing all neighbors).

Input: numCourses=4, prerequisites=[[1,0],[2,0],[3,1],[3,2]]  Output: [0,1,2,3]

## Dry Run (DFS Topological Sort)

Graph: 0→1, 0→2, 1→3, 2→3

| DFS | postorder adds | result (reversed) |
|-----|---------------|-------------------|
| dfs(0) → dfs(1) → dfs(3) | add 3 | [3] |
| back to dfs(1) | add 1 | [3,1] |
| back to dfs(0) → dfs(2) → dfs(3) done | add 2 | [3,1,2] |
| back to dfs(0) | add 0 | [3,1,2,0] |
| reverse: | [0,2,1,3] | valid topo order |

In [ ]:
class Solution:
    def findOrder(self, numCourses: int, prerequisites: List[List[int]]) -> List[int]:
        graph = defaultdict(list)
        for course, pre in prerequisites:
            graph[pre].append(course)

        state = [0] * numCourses   # 0=unvisited, 1=visiting, 2=done
        result = []

        def dfs(node):
            if state[node] == 1:
                return False       # cycle
            if state[node] == 2:
                return True
            state[node] = 1
            for neighbor in graph[node]:
                if not dfs(neighbor):
                    return False
            state[node] = 2
            result.append(node)    # postorder: add AFTER all neighbors done
            return True

        for i in range(numCourses):
            if not dfs(i):
                return []
        return result[::-1]        # reverse postorder = topological order

print(Solution().findOrder(4, [[1,0],[2,0],[3,1],[3,2]]))  # [0,2,1,3] or similar

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(V + E) | O(V + E) – graph + state + result |
| **Final: O(V + E)** | **Final: O(V + E)** |

# 7. LeetCode 417 – Pacific Atlantic Water Flow

Find cells from which water can flow to BOTH Pacific (top/left) and Atlantic (bottom/right).
Reverse: BFS/DFS from ocean edges inward.

Input: heights=[[1,2,2,3,5],[3,2,3,4,4],[2,4,5,3,1],[6,7,1,4,5],[5,1,1,2,4]]
Output: [[0,4],[1,3],[1,4],[2,2],[3,0],[3,1],[4,0]]

## Dry Run (Reverse BFS from ocean borders)

Pacific borders: top row + left column → BFS expanding to higher/equal neighbors
Atlantic borders: bottom row + right column → BFS expanding to higher/equal neighbors
Answer: cells reachable from BOTH

| pass | start cells | finds reachable from |
|------|-------------|---------------------|
| Pacific BFS | top row + left col | all cells water can reach pacific |
| Atlantic BFS | bottom row + right col | all cells water can reach atlantic |
| Intersection | both sets | answer |

In [ ]:
class Solution:
    def pacificAtlantic(self, heights: List[List[int]]) -> List[List[int]]:
        rows, cols = len(heights), len(heights[0])
        dirs = [(1,0),(-1,0),(0,1),(0,-1)]

        def bfs(starts):
            visited = set(starts)
            queue = deque(starts)
            while queue:
                r, c = queue.popleft()
                for dr, dc in dirs:
                    nr, nc = r+dr, c+dc
                    if (0<=nr<rows and 0<=nc<cols and
                            (nr,nc) not in visited and
                            heights[nr][nc] >= heights[r][c]):
                        visited.add((nr,nc))
                        queue.append((nr,nc))
            return visited

        pac_starts = [(r, 0) for r in range(rows)] + [(0, c) for c in range(cols)]
        atl_starts = [(r, cols-1) for r in range(rows)] + [(rows-1, c) for c in range(cols)]

        pac = bfs(pac_starts)
        atl = bfs(atl_starts)
        return [[r, c] for r, c in pac & atl]

heights = [[1,2,2,3,5],[3,2,3,4,4],[2,4,5,3,1],[6,7,1,4,5],[5,1,1,2,4]]
print(Solution().pacificAtlantic(heights))

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(m*n) – each cell visited twice (pac + atl BFS) | O(m*n) – visited sets |
| **Final: O(m*n)** | **Final: O(m*n)** |

# 8. LeetCode 684 – Redundant Connection (Union-Find)

Find the edge that creates a cycle. Return it.
Use Union-Find (Disjoint Set Union): if both endpoints already in same set → cycle!

Input: edges=[[1,2],[1,3],[2,3]]  Output: [2,3]

## Dry Run (Union-Find)

parent = [0,1,2,3] (each node is its own parent initially)

| edge | find(u) | find(v) | same? | action |
|------|---------|---------|-------|--------|
| [1,2] | 1 | 2 | no | union: parent[1]=2 |
| [1,3] | find(1)=2 | 3 | no | union: parent[2]=3 |
| [2,3] | find(2)=3 | 3 | **yes** | **CYCLE → return [2,3]** |

In [ ]:
class Solution:
    def findRedundantConnection(self, edges: List[List[int]]) -> List[int]:
        n = len(edges)
        parent = list(range(n + 1))
        rank = [0] * (n + 1)

        def find(x):
            if parent[x] != x:
                parent[x] = find(parent[x])   # path compression
            return parent[x]

        def union(x, y):
            px, py = find(x), find(y)
            if px == py:
                return False            # already connected = cycle
            if rank[px] < rank[py]:
                px, py = py, px
            parent[py] = px
            if rank[px] == rank[py]:
                rank[px] += 1
            return True

        for u, v in edges:
            if not union(u, v):
                return [u, v]

print(Solution().findRedundantConnection([[1,2],[1,3],[2,3]]))   # [2,3]
print(Solution().findRedundantConnection([[1,2],[2,3],[3,4],[1,4],[1,5]]))  # [1,4]

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n * alpha(n)) ~ O(n) – alpha is inverse Ackermann | O(n) – parent + rank arrays |
| **Final: O(n)** | **Final: O(n)** |

# 9. LeetCode 743 – Network Delay Time (Dijkstra's Algorithm)

Find minimum time for signal to reach all nodes from source k.
Return max of shortest paths, or -1 if any unreachable.

Input: times=[[2,1,1],[2,3,1],[3,4,1]], n=4, k=2  Output: 2

## Dry Run (Dijkstra from node 2)

Graph: 2→1(1), 2→3(1), 3→4(1)

min-heap: [(0, k=2)]  dist = {2:0}

| pop (dist, node) | neighbors | update dist | heap |
|------------------|-----------|-------------|------|
| (0, 2) | 1(+1=1), 3(+1=1) | dist[1]=1, dist[3]=1 | [(1,1),(1,3)] |
| (1, 1) | no neighbors | | [(1,3)] |
| (1, 3) | 4(+1=2) | dist[4]=2 | [(2,4)] |
| (2, 4) | no neighbors | | [] |
| max(dist.values()) = max(0,1,1,2) = **2** | | | |

In [ ]:
class Solution:
    def networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int:
        graph = defaultdict(list)
        for u, v, w in times:
            graph[u].append((v, w))

        dist = {k: 0}
        heap = [(0, k)]           # (distance, node)

        while heap:
            d, node = heapq.heappop(heap)
            if d > dist.get(node, float('inf')):
                continue           # outdated entry
            for neighbor, weight in graph[node]:
                new_dist = d + weight
                if new_dist < dist.get(neighbor, float('inf')):
                    dist[neighbor] = new_dist
                    heapq.heappush(heap, (new_dist, neighbor))

        return max(dist.values()) if len(dist) == n else -1

print(Solution().networkDelayTime([[2,1,1],[2,3,1],[3,4,1]], 4, 2))  # 2

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O((V + E) log V) – heap operations | O(V + E) – graph + dist dict |
| **Final: O((V+E) log V)** | **Final: O(V + E)** |

# 10. LeetCode 130 – Surrounded Regions

Capture all 'O' regions NOT connected to border. Replace captured 'O' with 'X'.
Strategy: DFS from border 'O' cells to mark them safe ('T'), then flip.

Input: [['X','X','X'],['X','O','X'],['X','X','X']]  Output: [['X','X','X'],['X','X','X'],['X','X','X']]

## Dry Run

Step 1: DFS from all border 'O' cells, mark them 'T' (temporarily safe)
Step 2: scan entire grid:
  - 'O' → 'X' (surrounded, flip)
  - 'T' → 'O' (safe, restore)
  - 'X' stays 'X'

| Step | action |
|------|--------|
| Check borders | no 'O' on border in example |
| Flip | center 'O' → 'X' (not connected to border) |
| Result | all 'X' |

In [ ]:
class Solution:
    def solve(self, board: List[List[str]]) -> None:
        if not board:
            return
        rows, cols = len(board), len(board[0])

        def dfs(r, c):
            if r < 0 or r >= rows or c < 0 or c >= cols or board[r][c] != 'O':
                return
            board[r][c] = 'T'         # temporarily mark as safe
            dfs(r+1,c); dfs(r-1,c); dfs(r,c+1); dfs(r,c-1)

        # Step 1: mark all border-connected 'O' as safe
        for r in range(rows):
            dfs(r, 0); dfs(r, cols-1)
        for c in range(cols):
            dfs(0, c); dfs(rows-1, c)

        # Step 2: flip remaining 'O' to 'X', restore 'T' to 'O'
        for r in range(rows):
            for c in range(cols):
                if board[r][c] == 'O':
                    board[r][c] = 'X'
                elif board[r][c] == 'T':
                    board[r][c] = 'O'

board = [['X','X','X'],['X','O','X'],['X','X','X']]
Solution().solve(board)
print(board)  # all X

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(m*n) – each cell visited at most twice | O(m*n) – DFS recursion stack |
| **Final: O(m*n)** | **Final: O(m*n)** |